# Работа с датасетами

In [25]:
import pandas as pd
#датасеты с предсказаниями моделей на валидационных и тестовых данных
df_lrnet = pd.read_csv('/kaggle/input/datasets/mariaspasyuk/labels-and-scores/lrnet.csv', index_col = 0)
df_resnet = pd.read_csv('/kaggle/input/datasets/mariaspasyuk/labels-and-scores/resnet.csv', index_col = 0)
df_eye = pd.read_csv('/kaggle/input/datasets/mariaspasyuk/labels-and-scores/eye.csv', index_col = 0)

In [26]:
#датасеты с предсказаниями на тестовых данных Хуавея
lrnet_hw = pd.read_csv('/kaggle/input/datasets/mariaspasyuk/models-hw/lrnet_hw.csv', index_col = 0)
resnet_eye_hw = pd.read_csv('/kaggle/working/resnet_eye_hw_correct.csv', index_col = 0)

In [27]:
df_resnet = df_resnet.sort_values(by='filename', ignore_index = True)
df_lrnet = df_lrnet.sort_values(by='filename', ignore_index = True)
df_eye = df_eye.sort_values(by='filename', ignore_index = True)

In [28]:
from functools import reduce
dfs = [df_resnet, df_lrnet, df_eye]
df_merged = reduce(lambda left, right: pd.merge(left, right, on='filename', how='inner'), dfs)

In [29]:
from functools import reduce
dfs_hw = [resnet_eye_hw, lrnet_hw]
df_merged_hw = reduce(lambda left, right: pd.merge(left, right, on='filename', how='inner'),dfs_hw)

In [30]:
df_merged_hw = df_merged_hw.rename(columns={'Score': 'lrnet_score'})

In [31]:
df_merged_hw = df_merged_hw.drop('lrnet_pred', axis=1)

In [32]:
df_merged = df_merged.rename(columns={'Score_y': 'lrnet_score'})
df_merged = df_merged.rename(columns={'Score': 'label'})

In [34]:
df_merged = df_merged.drop(['filename', 'resnet_pred', 'lrnet_pred', 'eye_pred'], axis = 1)

In [35]:
df_merged_hw.head()

,filename,eye_score,resnet_score,lrnet_score
0,b2659357f2ee8313f83821a20db1b0813226482c68f5e1...,0.827473,0.212164,1.000000
1,d78b121e5529bf2720638a688824b4503f5a61ffc23e2e...,0.282207,0.054097,0.888889
2,a914aea93739f9d1e8b5edba8875c09de133cf113d2756...,0.276776,0.058064,1.000000
3,079cab4e845c69779d60f9b535e3b8618022d41dc24a43...,0.791866,0.864058,1.000000
4,03957815a78dc315902d8228968d074aa115cf967f59a1...,0.805819,0.838464,1.000000


In [ ]:
import pandas as pd
df_labels = pd.read_csv('/kaggle/input/datasets/mariaspasyuk/huawei-test/d83a0ce6-dc87-46a6-9679-98a71cf91886.csv')

df_merged_hw = df_merged_hw.merge(df_labels[['obj_id', 'label']], left_on='filename', right_on='obj_id', how='left')

df_merged_hw = df_merged_hw.drop(columns=['obj_id'])
print(f"Добавлено лейблов: {df_merged_hw['label'].notna().sum()} из {len(df_merged_hw)}")
print(f"Не найдено лейблов: {df_merged_hw['label'].isna().sum()}")
print(df_merged_hw.head())

In [37]:
df_merged_hw.head()

,filename,eye_score,resnet_score,lrnet_score,label
0,b2659357f2ee8313f83821a20db1b0813226482c68f5e1...,0.827473,0.212164,1.000000,1
1,d78b121e5529bf2720638a688824b4503f5a61ffc23e2e...,0.282207,0.054097,0.888889,0
2,a914aea93739f9d1e8b5edba8875c09de133cf113d2756...,0.276776,0.058064,1.000000,0
3,079cab4e845c69779d60f9b535e3b8618022d41dc24a43...,0.791866,0.864058,1.000000,1
4,03957815a78dc315902d8228968d074aa115cf967f59a1...,0.805819,0.838464,1.000000,1


In [20]:
df_merged_hw.to_csv('test_hw_correct.csv')

In [ ]:
df_merged.to_csv('data_3models.csv')

# Обучение логистической регрессии

In [1]:
import pandas as pd
df_merged_hw = pd.read_csv('/kaggle/working/test_hw_correct.csv', index_col = 0)

In [2]:
df_merged_hw.head()

,filename,eye_score,resnet_score,lrnet_score,label
0,b2659357f2ee8313f83821a20db1b0813226482c68f5e1...,0.827473,0.212164,1.000000,1
1,d78b121e5529bf2720638a688824b4503f5a61ffc23e2e...,0.282207,0.054097,0.888889,0
2,a914aea93739f9d1e8b5edba8875c09de133cf113d2756...,0.276776,0.058064,1.000000,0
3,079cab4e845c69779d60f9b535e3b8618022d41dc24a43...,0.791866,0.864058,1.000000,1
4,03957815a78dc315902d8228968d074aa115cf967f59a1...,0.805819,0.838464,1.000000,1


In [3]:
df_merged = pd.read_csv('/kaggle/working/data_3models.csv', index_col = 0)

In [4]:
df_merged.head()

,eye_score,resnet_score,lrnet_score,label
0,0.277454,0.106156,0.250000,1
1,0.290915,0.071749,0.781250,0
2,0.301820,0.206305,0.866667,0
3,0.672197,0.927658,0.444444,1
4,0.331840,0.094613,1.000000,1


In [6]:
x = df_merged.drop('label', axis = 1)

In [7]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, df_merged['label'], test_size = 0.3, random_state = 42, stratify = df_merged['label'])

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter = 50)
model.fit(x_train, y_train)
final_pred = model.predict(x_test)

print("Метрики на тестовой выборке:")
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score, confusion_matrix, classification_report
print(f"Accuracy = {accuracy_score(y_test, final_pred)}, f1 = {f1_score(y_test, final_pred)}, precision = {precision_score(y_test, final_pred)}, recall = {recall_score(y_test, final_pred)}")

In [7]:
y_test = df_merged_hw['label']

In [10]:
print("Метрики на данных Хуавея: ")
x_test = df_merged_hw.drop(['label', 'filename'], axis=1)

y_pred = model.predict(x_test)
probs = model.predict_proba(x_test)[:, 1]

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score, confusion_matrix, classification_report
print(f"Accuracy = {accuracy_score(y_test, y_pred)}, f1 = {f1_score(y_test, y_pred)}, auc = {roc_auc_score(y_test, probs)} precision = {precision_score(y_test, y_pred)}, recall = {recall_score(y_test, y_pred)}")

Метрики на данных Хуавея: 
Accuracy = 0.6, f1 = 0.6363636363636364, auc = 0.7053571428571429 precision = 0.875, recall = 0.5


# Усреднение результатов

In [142]:
df_merged_hw['mean_score'] = ((df_merged_hw['resnet_score'] + df_merged_hw['eye_score'] + df_merged_hw['lrnet_score'])/3)
df_merged_hw['prediction'] = (df_merged_hw['mean_score'] >= 0.5).astype(int)

In [143]:
y_pred = df_merged_hw['prediction']
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score, confusion_matrix, classification_report
print(f"Accuracy = {accuracy_score(y_test, y_pred)}, f1 = {f1_score(y_test, y_pred)}, auc = {roc_auc_score(y_test, df_merged_hw['mean_score'])}, precision = {precision_score(y_test, y_pred)}, recall = {recall_score(y_test, y_pred)}")

Accuracy = 0.775, f1 = 0.8301886792452831, auc = 0.8779761904761905, precision = 0.88, recall = 0.7857142857142857


# Взвешивание результатов

In [ ]:
df_merged_hw['weighted_score'] = (df_merged_hw['resnet_score']*0.2 + df_merged_hw['eye_score']*0.4 + df_merged_hw['lrnet_score']*0.4)
df_merged_hw['prediction'] = (df_merged_hw['weighted_score'] >= 0.5).astype(int)

y_pred = df_merged_hw['prediction']
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score, confusion_matrix, classification_report
print(f"Accuracy = {accuracy_score(y_test, y_pred)}, f1 = {f1_score(y_test, y_pred)}, auc = {roc_auc_score(y_test, df_merged_hw['weighted_score'])}, precision = {precision_score(y_test, y_pred)}, recall = {recall_score(y_test, y_pred)}")

# Голосование моделей

In [15]:
df_merged_hw['prediction'] = ((df_merged_hw[['resnet_score', 'lrnet_score', 'eye_score']] > 0.5).sum(axis=1) >= 2).astype(int)

y_pred = df_merged_hw['prediction']
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score, confusion_matrix, classification_report
print(f"Accuracy = {accuracy_score(y_test, y_pred)}, f1 = {f1_score(y_test, y_pred)}, auc = {roc_auc_score(y_test, df_merged_hw['prediction'])}, precision = {precision_score(y_test, y_pred)}, recall = {recall_score(y_test, y_pred)}")

Accuracy = 0.75, f1 = 0.8076923076923077, auc = 0.75, precision = 0.875, recall = 0.75
